Helsinki-NLP/opus-mt-ko-en  
https://huggingface.co/Helsinki-NLP/opus-mt-ko-en

In [17]:
from transformers import MarianMTModel, MarianTokenizer
import re

In [18]:
# 모델 로드
model_name = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

In [44]:
# 한국어-영어 코드스위칭 번역 함수
def translate_korean_phrases(text):
    # 영어 기준으로 분리
    phrases = re.split(r'([a-zA-Z0-9]+)', text)
    translated_phrases = []

    for phrase in phrases:
        # 한국어와 영어를 기준으로 구분
        if re.search(r'[가-힣]', phrase):  # 한국어가 포함된 구
            # 번역 수행
            inputs = tokenizer(phrase, return_tensors="pt", padding=True, truncation=True)
            outputs = model.generate(**inputs)
            translated_phrase = tokenizer.decode(outputs[0], skip_special_tokens=True)
            translated_phrases.append(translated_phrase)
        else:  # 영어 구나 기타 문장은 그대로 유지
            translated_phrases.append(phrase)

    # 공백만 있는 요소 제거
    translated_phrases = [phrase for phrase in translated_phrases if phrase.strip()]

    # 번역된 구문 결합
    return ' '.join(translated_phrases)

In [51]:
# 문장부호와 단어 사이에 불필요한 공백을 제거 함수
def clean_punctuation_spacing(text):
    text = re.sub(r'\s([.,!?\'":;])', r'\1', text) # 문장부호 앞의 공백 제거
    text = re.sub(r'([.,!?\'":;])([.,!?\'":;])', r'\1 \2', text) # 문장부호 연속 사이에 공백 삽입
    text = re.sub(r'([.,!?\'":;])\s?', r'\1 ', text) # 문장부호 뒤에 적절한 공백 추가
    text = re.sub(r'\s+', ' ', text) # 중복 공백 제거
    return text.strip()

In [46]:
# 테스트
input_text = "오늘 아침에는 정말 바빴어. I had to finish some 보고서 for my class, but then I realized I forgot to bring my 노트북. 그래서 친구한테 연락했는데, she said, 'Don't worry, you can use mine.' 그래서 겨우 작업을 끝내고 학교에 갔는데, during the presentation, the 교수님 asked me a really tricky question in English, and I was like, 'Oh no, I need to think about this for a second.' 다행히 잘 대답해서 무사히 넘어갔어."

In [53]:
# 함수 실행
translated_text = translate_korean_phrases(input_text)
cleaned_text = clean_punctuation_spacing(translated_text)
print("Translated Text:", cleaned_text)

Translated Text: I was really busy this morning. I had to finish some Report for my class, but then I realized I forgot to bring my So I called my friend. she said, ' Don' t worry, you can use mine So I just finished my job and went to school. during the presentation, the Professor. asked me a really tricky question in English, and I was like, ' Oh no, I need to think about this for a second I' m glad he answered well and made it through.
